# Experiment 5 — EHO Parameter Sensitivity

Khảo sát riêng ảnh hưởng của α, β, số clan (giữ tổng quần thể bằng 80) và số voi tách đàn. Mỗi cấu hình chạy trên 5 seed cố định `[11, 23, 37, 53, 71]`, với ngân sách tối đa 4.000 objective evaluations. Các biểu đồ thể hiện trung bình ± độ lệch chuẩn mẫu của độ dài tour tốt nhất.

Notebook dùng public API `EHOConfig`/`solve`, tìm đường dẫn tương đối từ repo root hoặc thư mục `Experiment/`, giới hạn tối đa **75 runs** và **120 giây** tổng thời gian; mỗi lần gọi solver có time limit hợp tác tối đa 10 giây (hoặc phần thời gian còn lại).

In [ ]:
from pathlib import Path
import html
import math
import statistics
import sys
from time import perf_counter

# Works from the repository root or when Jupyter starts in Experiment/.
ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ga_eho_tsp.data.matrix_loader import load_matrix
from ga_eho_tsp.domain.models import EHOConfig, GAConfig
from ga_eho_tsp.algorithms.eho import solve as solve_eho
from ga_eho_tsp.algorithms.ga import solve as solve_ga
from IPython.display import display, Markdown, SVG

MODE = "QUICK"
SEEDS = [11, 23, 37, 53, 71]
TOTAL_TIME_CAP_SECONDS = 120.0
PER_RUN_TIME_CAP_SECONDS = 10.0
NOTEBOOK_START = perf_counter()
RUN_COUNT = 0
MAX_RUNS = 75  # E5: 15 configurations x 5 seeds; E6 overrides to 70.
RUN_RESULTS = []
RESULT_CACHE = {}

def show_table(headers, rows):
    text = "| " + " | ".join(headers) + " |\n"
    text += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    text += "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)
    display(Markdown(text))

def run_with_caps(problem, algorithm, config, tag):
    """Run only within the declared notebook run/time caps; solver has its own cooperative cap."""
    global RUN_COUNT
    if tag in RESULT_CACHE:
        return RESULT_CACHE[tag]
    elapsed = perf_counter() - NOTEBOOK_START
    if RUN_COUNT >= MAX_RUNS or elapsed >= TOTAL_TIME_CAP_SECONDS:
        return None
    remaining = TOTAL_TIME_CAP_SECONDS - elapsed
    config = __import__("dataclasses").replace(
        config, time_limit_seconds=min(PER_RUN_TIME_CAP_SECONDS, remaining)
    )
    RUN_COUNT += 1
    result = algorithm(problem, config)
    assert result.is_valid
    assert problem.route_length(result.best_route) == result.best_distance
    RUN_RESULTS.append((tag, result))
    RESULT_CACHE[tag] = result
    return result

def mean_std(values):
    if not values:
        return None, None
    return statistics.mean(values), (statistics.stdev(values) if len(values) > 1 else 0.0)

def plot_bars(title, labels, means, stds, ylabel="Best tour distance"):
    available = [(label, mean, std) for label, mean, std in zip(labels, means, stds)
                 if mean is not None and std is not None]
    if not available:
        display(Markdown(f"**{title}:** chưa có run hợp lệ trong giới hạn hiện tại. Restart kernel và chạy Run All để bắt đầu batch mới."))
        return
    labels, means, stds = zip(*available)
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(max(7, len(labels) * 1.15), 4.5))
        ax.bar(range(len(labels)), means, yerr=stds, capsize=4, color="#4c8c78")
        ax.set_xticks(range(len(labels)), labels, rotation=25, ha="right")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=.25)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
    except ImportError:
        # SVG keeps the requested chart available in a notebook kernel without matplotlib.
        width, height = max(640, 90 * len(labels)), 390
        left, top, bottom = 55, 35, 105
        plot_h, plot_w = height - top - bottom, width - left - 20
        peak = max((m + s for m, s in zip(means, stds)), default=1) or 1
        pieces = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
                  '<rect width="100%" height="100%" fill="white"/>',
                  f'<text x="{width/2}" y="22" text-anchor="middle" font-size="16">{html.escape(title)}</text>',
                  f'<line x1="{left}" y1="{top}" x2="{left}" y2="{height-bottom}" stroke="#333"/>',
                  f'<line x1="{left}" y1="{height-bottom}" x2="{width-15}" y2="{height-bottom}" stroke="#333"/>']
        slot = plot_w / max(1, len(labels)); bar_w = slot * .56
        for i, (label, mean, std) in enumerate(zip(labels, means, stds)):
            x = left + slot*i + (slot-bar_w)/2
            bar_h = mean / peak * plot_h
            y = height-bottom-bar_h
            mid = x + bar_w/2
            err = std / peak * plot_h
            pieces.extend([f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" fill="#4c8c78"/>',
                           f'<line x1="{mid:.1f}" y1="{y-err:.1f}" x2="{mid:.1f}" y2="{y+err:.1f}" stroke="#222" stroke-width="2"/>',
                           f'<line x1="{mid-6:.1f}" y1="{y-err:.1f}" x2="{mid+6:.1f}" y2="{y-err:.1f}" stroke="#222"/>',
                           f'<line x1="{mid-6:.1f}" y1="{y+err:.1f}" x2="{mid+6:.1f}" y2="{y+err:.1f}" stroke="#222"/>',
                           f'<text x="{mid:.1f}" y="{height-bottom+20}" text-anchor="middle" font-size="12">{html.escape(str(label))}</text>'])
        pieces.append('</svg>')
        display(SVG("".join(pieces)))

## Thiết lập và dữ liệu

E5.1–E5.2 dùng `data/15Cities-1194.txt`; E5.3–E5.4 dùng `data/26Cities-937.txt`. Mỗi cell sweep tự nạp lại dataset và cấu hình baseline, nên có thể chạy lại riêng từng sweep sau khi đã chạy cell import/helpers.


In [ ]:
DATA_15 = load_matrix(ROOT / "data" / "15Cities-1194.txt")
DATA_26 = load_matrix(ROOT / "data" / "26Cities-937.txt")
EVALUATION_BUDGET = 4000
BASE = EHOConfig(max_evaluations=EVALUATION_BUDGET, max_iterations=250, iterations=250)
print(f"Root: {ROOT}; seed mode: {MODE}; seeds={SEEDS}; budget={EVALUATION_BUDGET}; run cap={MAX_RUNS}")

## E5.1 — α

Sweep `α ∈ {0.1, 0.3, 0.5, 0.7, 0.9}` trên bộ 15 thành phố.

In [ ]:
DATA_15 = load_matrix(ROOT / "data" / "15Cities-1194.txt")
BASE = EHOConfig(max_evaluations=4000, max_iterations=250, iterations=250)
alpha_values = [0.1, 0.3, 0.5, 0.7, 0.9]
alpha_runs = {}
for alpha in alpha_values:
    alpha_runs[alpha] = []
    for seed in SEEDS:
        result = run_with_caps(DATA_15, solve_eho,
            __import__("dataclasses").replace(BASE, alpha=alpha, seed=seed), ("alpha", alpha, seed))
        if result is None: break
        alpha_runs[alpha].append(result)
alpha_summary = []
for a, rs in alpha_runs.items():
    mean, std = mean_std([r.best_distance for r in rs])
    alpha_summary.append((a, len(rs), round(mean, 3) if mean is not None else "—",
                          round(std, 3) if std is not None else "—"))
show_table(["α", "runs", "mean", "std"], alpha_summary)
plot_bars("E5.1 — Sensitivity to α (15-city)", [str(x) for x in alpha_values],
          [mean_std([r.best_distance for r in alpha_runs[x]])[0] if alpha_runs[x] else None for x in alpha_values],
          [mean_std([r.best_distance for r in alpha_runs[x]])[1] if alpha_runs[x] else None for x in alpha_values])

## E5.2 — β

Sweep `β ∈ {0.1, 0.3, 0.5, 0.7}` trên bộ 15 thành phố.

In [ ]:
DATA_15 = load_matrix(ROOT / "data" / "15Cities-1194.txt")
BASE = EHOConfig(max_evaluations=4000, max_iterations=250, iterations=250)
beta_values = [0.1, 0.3, 0.5, 0.7]
beta_runs = {}
for beta in beta_values:
    beta_runs[beta] = []
    for seed in SEEDS:
        result = run_with_caps(DATA_15, solve_eho,
            __import__("dataclasses").replace(BASE, beta=beta, seed=seed), ("beta", beta, seed))
        if result is None: break
        beta_runs[beta].append(result)
show_table(["β", "runs", "mean", "std"], [(b, len(rs), round(mean_std([r.best_distance for r in rs])[0], 3) if rs else "—",
    round(mean_std([r.best_distance for r in rs])[1], 3) if rs else "—") for b, rs in beta_runs.items()])
plot_bars("E5.2 — Sensitivity to β (15-city)", [str(x) for x in beta_values],
          [mean_std([r.best_distance for r in beta_runs[x]])[0] if beta_runs[x] else None for x in beta_values],
          [mean_std([r.best_distance for r in beta_runs[x]])[1] if beta_runs[x] else None for x in beta_values])

## E5.3 — Số clan, tổng quần thể cố định

So sánh 2, 4 và 8 clan trên bộ 26 thành phố. Số voi mỗi clan lần lượt là 40, 20 và 10, nên tổng quần thể luôn bằng 80.

In [ ]:
DATA_26 = load_matrix(ROOT / "data" / "26Cities-937.txt")
BASE = EHOConfig(max_evaluations=4000, max_iterations=250, iterations=250)
clan_values = [2, 4, 8]
clan_runs = {}
for clans in clan_values:
    elephants = 80 // clans
    assert clans * elephants == 80
    clan_runs[clans] = []
    for seed in SEEDS:
        result = run_with_caps(DATA_26, solve_eho,
            __import__("dataclasses").replace(BASE, clan_count=clans, elephants_per_clan=elephants, seed=seed),
            ("clan_count", clans, seed))
        if result is None: break
        clan_runs[clans].append(result)
show_table(["clans", "elephants/clan", "population", "runs", "mean", "std"],
    [(c, 80//c, 80, len(rs), round(mean_std([r.best_distance for r in rs])[0], 3) if rs else "—",
      round(mean_std([r.best_distance for r in rs])[1], 3) if rs else "—") for c, rs in clan_runs.items()])
plot_bars("E5.3 — Sensitivity to clan count (26-city)", [str(x) for x in clan_values],
          [mean_std([r.best_distance for r in clan_runs[x]])[0] if clan_runs[x] else None for x in clan_values],
          [mean_std([r.best_distance for r in clan_runs[x]])[1] if clan_runs[x] else None for x in clan_values])

## E5.4 — Separation count

Thử `separation_count_per_clan ∈ {0, 1, 2}` trên bộ 26 thành phố.

In [ ]:
DATA_26 = load_matrix(ROOT / "data" / "26Cities-937.txt")
BASE = EHOConfig(max_evaluations=4000, max_iterations=250, iterations=250)
separation_values = [0, 1, 2]
separation_runs = {}
for count in separation_values:
    separation_runs[count] = []
    for seed in SEEDS:
        result = run_with_caps(DATA_26, solve_eho,
            __import__("dataclasses").replace(BASE, separation_count_per_clan=count, seed=seed),
            ("separation_count", count, seed))
        if result is None: break
        separation_runs[count].append(result)
show_table(["separation/clan", "runs", "mean", "std"],
    [(c, len(rs), round(mean_std([r.best_distance for r in rs])[0], 3) if rs else "—",
      round(mean_std([r.best_distance for r in rs])[1], 3) if rs else "—") for c, rs in separation_runs.items()])
plot_bars("E5.4 — Sensitivity to separation count (26-city)", [str(x) for x in separation_values],
          [mean_std([r.best_distance for r in separation_runs[x]])[0] if separation_runs[x] else None for x in separation_values],
          [mean_std([r.best_distance for r in separation_runs[x]])[1] if separation_runs[x] else None for x in separation_values])
print(f"Runs completed: {RUN_COUNT}/{MAX_RUNS}; elapsed notebook time: {perf_counter()-NOTEBOOK_START:.2f}s")